# Exercise 1: Factory Method for payment system

First we will create an abstract PaymentProcessor. We do this because all payment types share the same behavior (validate → calculate fee → process). The abstract class defines this contract.

Then we create concrete processors because each payment type has its own validation rules and fee structure. Separate class = separate responsibility.

In [1]:
from abc import ABC, abstractmethod


# Abstract base class
class PaymentProcessor(ABC):

    @abstractmethod
    def validate(self, details: dict) -> bool:
        pass

    @abstractmethod
    def calculate_fee(self, amount: float) -> float:
        pass

    def process(self, amount: float, details: dict) -> dict:
        validation_error = self.validate(details)
        if validation_error:
            return {"success": False, "error": validation_error}

        fee = self.calculate_fee(amount)
        total = amount + fee
        return {"success": True, "method": self.name, "amount": total, "fee": fee}


# Concrete implementations
class CreditCardProcessor(PaymentProcessor):
    name = "credit_card"

    def validate(self, details: dict):
        card_number = details.get("card_number")
        cvv = details.get("cvv")

        if not card_number or len(card_number) != 16:
            return "Invalid card number"
        if not cvv or len(cvv) != 3:
            return "Invalid CVV"
        return None

    def calculate_fee(self, amount: float) -> float:
        return amount * 0.029


class BankTransferProcessor(PaymentProcessor):
    name = "bank_transfer"

    def validate(self, details: dict):
        iban = details.get("iban")

        if not iban or len(iban) < 15:
            return "Invalid IBAN"
        return None

    def calculate_fee(self, amount: float) -> float:
        return 1.50


class PayPalProcessor(PaymentProcessor):
    name = "paypal"

    def validate(self, details: dict):
        email = details.get("email")

        if not email or "@" not in email:
            return "Invalid PayPal email"
        return None

    def calculate_fee(self, amount: float) -> float:
        return amount * 0.034 + 0.30


# Factory
class PaymentFactory:
    _processors = {
        "credit_card": CreditCardProcessor,
        "bank_transfer": BankTransferProcessor,
        "paypal": PayPalProcessor,
    }

    @classmethod
    def get_processor(cls, payment_type: str) -> PaymentProcessor:
        processor_class = cls._processors.get(payment_type)
        if not processor_class:
            raise ValueError(f"Unknown payment type: {payment_type}")
        return processor_class()

    @classmethod
    def register_processor(cls, payment_type: str, processor_class):
        cls._processors[payment_type] = processor_class


# Usage
if __name__ == "__main__":
    factory = PaymentFactory()

    # Credit card
    processor = factory.get_processor("credit_card")
    result = processor.process(100.0, {"card_number": "1234567890123456", "cvv": "123"})
    print(result)

    # Bank transfer
    processor = factory.get_processor("bank_transfer")
    result = processor.process(100.0, {"iban": "FR7630006000011234567890189"})
    print(result)

    # PayPal
    processor = factory.get_processor("paypal")
    result = processor.process(100.0, {"email": "user@example.com"})
    print(result)

{'success': True, 'method': 'credit_card', 'amount': 102.9, 'fee': 2.9000000000000004}
{'success': True, 'method': 'bank_transfer', 'amount': 101.5, 'fee': 1.5}
{'success': True, 'method': 'paypal', 'amount': 103.7, 'fee': 3.7}


In [7]:
#Expected Usage after Refactoring

factory = PaymentFactory()
processor = factory.get_processor("credit_card")
result = processor.process(100.0, {"card_number": "1234567890123456", "cvv": "123"})

# Exercise 2: Builder Pattern for HR Employee Onboarding system

In [2]:
class Employee:
    def __init__(self, data: dict):
        self.first_name = data.get("first_name")
        self.last_name = data.get("last_name")
        self.email = data.get("email")
        self.department = data.get("department")
        self.position = data.get("position")
        self.salary = data.get("salary")
        self.start_date = data.get("start_date")
        self.manager_id = data.get("manager_id")
        self.phone = data.get("phone")
        self.address = data.get("address")
        self.has_parking = data.get("has_parking", False)
        self.has_laptop = data.get("has_laptop", False)
        self.has_vpn_access = data.get("has_vpn_access", False)
        self.has_admin_rights = data.get("has_admin_rights", False)
        self.contract_type = data.get("contract_type", "permanent")

    def __repr__(self):
        return f"Employee({self.first_name} {self.last_name} - {self.position})"


class EmployeeBuilder:
    def __init__(self):
        self._data = {}

    def with_name(self, first_name: str, last_name: str):
        self._data["first_name"] = first_name
        self._data["last_name"] = last_name
        return self

    def with_email(self, email: str):
        self._data["email"] = email
        return self

    def with_contact(self, phone: str = None, address: str = None):
        self._data["phone"] = phone
        self._data["address"] = address
        return self

    def with_job(self, department: str, position: str, salary: float, start_date: str = None):
        self._data["department"] = department
        self._data["position"] = position
        self._data["salary"] = salary
        self._data["start_date"] = start_date
        return self

    def with_manager(self, manager_id: int):
        self._data["manager_id"] = manager_id
        return self

    def with_equipment(self, laptop: bool = False, parking: bool = False):
        self._data["has_laptop"] = laptop
        self._data["has_parking"] = parking
        return self

    def with_access(self, vpn: bool = False, admin: bool = False):
        self._data["has_vpn_access"] = vpn
        self._data["has_admin_rights"] = admin
        return self

    def with_contract(self, contract_type: str):
        self._data["contract_type"] = contract_type
        return self

    def build(self) -> Employee:
        # Validation
        if not self._data.get("first_name") or not self._data.get("last_name"):
            raise ValueError("Name is required")
        if not self._data.get("email") or "@" not in self._data["email"]:
            raise ValueError("Valid email is required")

        return Employee(self._data)


# Preset builders
class DeveloperBuilder(EmployeeBuilder):
    def __init__(self, first_name: str, last_name: str, email: str):
        super().__init__()
        self.with_name(first_name, last_name)
        self.with_email(email)
        self.with_equipment(laptop=True, parking=False)
        self.with_access(vpn=True, admin=True)
        self.with_contract("permanent")


class InternBuilder(EmployeeBuilder):
    def __init__(self, first_name: str, last_name: str, email: str):
        super().__init__()
        self.with_name(first_name, last_name)
        self.with_email(email)
        self.with_equipment(laptop=True, parking=False)
        self.with_access(vpn=False, admin=False)
        self.with_contract("internship")


# Usage
if __name__ == "__main__":
    # Full builder chain
    employee = (
        EmployeeBuilder()
        .with_name("John", "Doe")
        .with_email("john.doe@company.com")
        .with_job("Engineering", "Senior Developer", 75000, "2024-01-15")
        .with_contact(phone="+33612345678")
        .with_equipment(laptop=True)
        .with_access(vpn=True, admin=True)
        .build()
    )
    print(employee)

    # Preset developer
    dev = DeveloperBuilder("Alice", "Martin", "alice@company.com").with_job("Engineering", "Backend Dev", 65000).build()
    print(dev)

    # Preset intern
    intern = InternBuilder("Bob", "Smith", "bob@company.com").with_job("Marketing", "Intern", 15000).with_manager(42).build()
    print(intern)

Employee(John Doe - Senior Developer)
Employee(Alice Martin - Backend Dev)
Employee(Bob Smith - Intern)


In [8]:
# Expected usage after Refactoring

employee = (
    EmployeeBuilder()
    .with_name("John", "Doe")
    .with_email("john.doe@company.com")
    .with_job("Engineering", "Senior Developer", 75000)
    .with_equipment(laptop=True, parking=False)
    .with_access(vpn=True, admin=True)
    .build()
)

# Or use a preset
dev = DeveloperBuilder("John", "Doe", "john.doe@company.com").build()

# Exercise 3: Singleton Pattern for configuration manager

In [6]:
import json

class ConfigManager:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._load_config()
        return cls._instance

    def _load_config(self):
        try:
            with open("config.json", "r") as f:
                self._config = json.load(f)
        except FileNotFoundError:
            # Default config if file is missing
            self._config = {
                "app": {
                    "name": "MyApp",
                    "debug": True
                },
                "database": {
                    "host": "localhost",
                    "port": 5432
                },
                "email": {
                    "smtp_server": "smtp.gmail.com",
                    "from_address": "noreply@myapp.com"
                },
                "payment": {
                    "currency": "EUR",
                    "provider": "stripe"
                }
            }

    @classmethod
    def get_instance(cls):
        if cls._instance is None:
            cls._instance = cls()
        return cls._instance

    def get(self, key):
        keys = key.split(".")
        value = self._config
        for k in keys:
            value = value[k]
        return value


# ----------- Services using Singleton -----------

def start_application():
    config = ConfigManager.get_instance()
    app_name = config.get("app.name")
    debug = config.get("app.debug")
    print(f"Starting {app_name} (debug={debug})")


def connect_database():
    config = ConfigManager.get_instance()
    db_host = config.get("database.host")
    db_port = config.get("database.port")
    print(f"Connecting to database at {db_host}:{db_port}")


def send_email(to, subject):
    config = ConfigManager.get_instance()
    smtp = config.get("email.smtp_server")
    sender = config.get("email.from_address")
    print(f"Sending email from {sender} via {smtp} to {to}: {subject}")


def process_payment(amount):
    config = ConfigManager.get_instance()
    currency = config.get("payment.currency")
    provider = config.get("payment.provider")
    print(f"Processing {amount} {currency} via {provider}")


# ----------- Usage -----------

if __name__ == "__main__":
    start_application()
    connect_database()
    send_email("user@test.com", "Welcome")
    process_payment(99.99)

    # Proof: same instance
    config1 = ConfigManager.get_instance()
    config2 = ConfigManager.get_instance()
    print(f"\nSame instance? {config1 is config2}")  # True

Starting MyApp (debug=True)
Connecting to database at localhost:5432
Sending email from noreply@myapp.com via smtp.gmail.com to user@test.com: Welcome
Processing 99.99 EUR via stripe

Same instance? True


In [9]:
#Expected usage after Refactoring

# First call loads the file
config = ConfigManager.get_instance()

# Subsequent calls return same instance (no file read)
config = ConfigManager.get_instance()

# Access values
db_host = config.get("database.host")
debug = config.get("app.debug")